# modeling_v13 — 2차 선별: **Conservative + RFECV**

**목적**: M1 다이어트 `Conservative` 셋에 RFECV 2차 선별을 얹어, 최종 OOF로 비교.
**RFECV(수동)** — prunable 피처를 gain 하위부터 step(5)개씩 제거하며 GroupKFold(C20) OOF 곡선을 그려 **최소 RMSE 크기**를 고른다.

**고정(항상 포함)**: core10(10) + 필수 5센서 champion → floor(5센서≥1) 보장·백본.
**가변(prunable)**: `Conservative` diet − 고정. 여기서만 RFECV가 부분집합을 고른다.
**선별 목적함수**: `GroupKFold(C20)` OOF (신규 Lot 정직-CV) — 프로젝트 결정지표.
**최종 보고**: 고른 셋 + baseline(2차선별 없음)을 `M8_PARAMS`·705로 **KFold + GroupKFold** 둘 다.

> ⚠️ 전제: `v13_select_common.py`, `../modeling_v8/v8_timeline_common.py`,
> `../문제1(하)/train_data.csv`, `../pm_log.json`, `feature_diet_selected.json`,
> `data/v13_fdc_pool_wf_oof.csv.gz` 가 제자리에 있어야 함. **CPU 전용(GPU 이득 없음).**
> 4개 노트북(Cons/Bal × RFECV/GA)은 서로 독립 — **병렬 실행 가능**.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import json, pandas as pd
import v13_select_common as sc

PRESET = "Conservative"
METHOD = "RFECV"

## 1. 로드 & 피처 그룹 (고정/가변)

In [2]:
df, y, groups, g = sc.load(PRESET)
ok, have = sc.floor_ok(g["fixed"])
print("df:", df.shape)
print(f"fixed(core10+champion)={len(g['fixed'])}  prunable(diet-fixed)={len(g['prunable'])}")
print("fixed floor:", ok, have)

df: (11939, 667)
fixed(core10+champion)=15  prunable(diet-fixed)=136
fixed floor: True {'C17': 1, 'C11': 1, 'C31': 1, 'C15': 1, 'C16': 1}


## 2. RFECV 선별

**RFECV(수동)** — prunable 피처를 gain 하위부터 step(5)개씩 제거하며 GroupKFold(C20) OOF 곡선을 그려 **최소 RMSE 크기**를 고른다.

In [3]:
# RFECV 선별 (step=5, min_keep=10, 5-fold). 런타임 ~2–3분.
res = sc.rfe_cv(df, y, groups, g["fixed"], g["prunable"],
                step=5, min_keep=10, n_splits=5, verbose=True)
selected = res["best_subset"]
print(f"\nRFECV 선택: prunable {len(g['prunable'])} -> {len(selected)}  (best GKF={res['best_rmse']:.3f})")

  RFE  prunable=136  GKF_RMSE=69.592
  RFE  prunable=131  GKF_RMSE=69.456
  RFE  prunable=126  GKF_RMSE=69.586
  RFE  prunable=121  GKF_RMSE=69.483
  RFE  prunable=116  GKF_RMSE=69.820
  RFE  prunable=111  GKF_RMSE=69.599
  RFE  prunable=106  GKF_RMSE=69.616
  RFE  prunable=101  GKF_RMSE=69.782
  RFE  prunable= 96  GKF_RMSE=69.791
  RFE  prunable= 91  GKF_RMSE=69.767
  RFE  prunable= 86  GKF_RMSE=69.607
  RFE  prunable= 81  GKF_RMSE=69.813
  RFE  prunable= 76  GKF_RMSE=69.709
  RFE  prunable= 71  GKF_RMSE=69.858
  RFE  prunable= 66  GKF_RMSE=69.818
  RFE  prunable= 61  GKF_RMSE=69.967
  RFE  prunable= 56  GKF_RMSE=69.937
  RFE  prunable= 51  GKF_RMSE=70.246
  RFE  prunable= 46  GKF_RMSE=70.100
  RFE  prunable= 41  GKF_RMSE=69.979
  RFE  prunable= 36  GKF_RMSE=69.978
  RFE  prunable= 31  GKF_RMSE=70.283
  RFE  prunable= 26  GKF_RMSE=70.385
  RFE  prunable= 21  GKF_RMSE=70.435
  RFE  prunable= 16  GKF_RMSE=70.505
  RFE  prunable= 11  GKF_RMSE=70.814
  RFE  prunable= 10  GKF_RMSE=71.235



## 3. 최종 OOF — 선택셋 vs baseline(2차선별 없음) · M8_PARAMS·705

In [4]:
baseline = sc.final_report(df, y, groups, g["fixed"], g["prunable"], f"{PRESET}+diet(no-2nd)")
best     = sc.final_report(df, y, groups, g["fixed"], selected,       f"{PRESET}+{METHOD}")
tbl = pd.DataFrame([baseline, best])[["label","n_total","n_selected",
                                      "KFold_OOF","GroupKFold_C20","floor_ok"]]
print(tbl.to_string(index=False))
tbl

                    label  n_total  n_selected  KFold_OOF  GroupKFold_C20  floor_ok
Conservative+diet(no-2nd)      151         136     51.817          71.192      True
       Conservative+RFECV      146         131     51.671          71.194      True


,label,n_total,n_selected,KFold_OOF,GroupKFold_C20,floor_ok
0,Conservative+diet(no-2nd),151,136,51.817,71.192,True
1,Conservative+RFECV,146,131,51.671,71.194,True


## 4. floor 재확인 & 선택 피처

In [5]:
okb, haveb = sc.floor_ok(best["features"])
assert okb, "FLOOR VIOLATION"
print("final floor:", okb, haveb)
print(f"selected prunable ({len(selected)}):")
for f in sorted(selected):
    print("  ", f)

final floor: True {'C17': 4, 'C11': 12, 'C31': 3, 'C15': 3, 'C16': 3}
selected prunable (131):
   C11_last_step1
   C11_last_step6
   C11_last_step7
   C11_max_step1
   C11_max_step4
   C11_max_step6
   C11_max_step7
   C11_mean_step5
   C11_min_step1
   C11_std_step6
   C11_std_step7
   C12_mean_step1
   C15_last_step4
   C15_max_step4
   C16_mean_step4
   C16_mean_step5
   C17_min_step1
   C17_std_step1
   C17_std_step6
   C18_last_step1
   C18_last_step4
   C18_max_step4
   C18_mean_step1
   C18_mean_step5
   C18_min_step1
   C18_min_step4
   C18_min_step6
   C18_min_step7
   C18_std_step6
   C18_std_step7
   C25_last_step1
   C25_last_step4
   C25_last_step6
   C25_last_step7
   C25_max_step1
   C25_max_step6
   C25_mean_step4
   C25_mean_step5
   C25_min_step4
   C25_min_step7
   C25_std_step1
   C25_std_step4
   C25_std_step6
   C25_std_step7
   C27_last_step4
   C27_max_step4
   C27_std_step4
   C31_max_step4
   C31_mean_step5
   C32_last_step4
   C32_mean_step5
   C32_min_step4

## 5. 산출물 저장 (4개 노트북 결과 통합용)

In [6]:
out = {"preset": PRESET, "method": METHOD,
       "baseline": {k: baseline[k] for k in ["n_total","KFold_OOF","GroupKFold_C20"]},
       "selected_result": {k: best[k] for k in ["n_total","n_selected","KFold_OOF","GroupKFold_C20","floor_ok"]},
       "selected_features": sorted(selected)}
fn = f"select_result_{PRESET}_{METHOD}.json"
json.dump(out, open(fn, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("saved:", fn)

saved: select_result_Conservative_RFECV.json


---
### 4개 노트북 통합 비교 (모두 실행 후)
각 노트북이 `select_result_<preset>_<method>.json` 을 남긴다. 4개를 모아 GroupKFold(C20) 기준으로
최저 RMSE 조합을 고르면 된다 (동률 시 더 슬림한 `n_total` 우선).